# 实验5 视觉动作识别 · Kaggle 正式训练

- 数据：`basketball-action-clips`（7424 clip = 6224 动作 + 1200 背景，16帧×112px，13 动作类 + `_none`）
- 双档：`mc3_18`（部署档，11.6M 参数）+ `r2plus1d_18`（精度档，31M），Kinetics-400 预训练微调
- 预计总时长约 1 小时（mc3 约 15 分钟 + r2plus1d 约 45 分钟 + 导出 5 分钟）

**Notebook 设置：Accelerator = GPU T4（单卡即可，不需要 T4×2），Internet = On**（pip 装 onnx 依赖用）。

In [ ]:
!pip -q install onnx onnxruntime onnxscript 2>&1 | tail -1

In [ ]:
import shutil
from pathlib import Path

ds_candidates = [p for p in Path('/kaggle/input').iterdir()
                 if (p / 'action_clips' / 'meta.json').exists()]
assert ds_candidates, '未找到含 action_clips 的数据集：请在右侧 Add Input 挂载 basketball-action-clips'
DS = ds_candidates[0]
DATA = DS / 'action_clips'
WORK = Path('/kaggle/working')
(WORK / 'scripts').mkdir(exist_ok=True)
for f in DS.glob('*.py'):
    shutil.copy(f, WORK / 'scripts' / f.name)
print('DATA =', DATA)
print('scripts =', sorted(p.name for p in (WORK / 'scripts').glob('*.py')))

In [ ]:
# 训练双档（逐 epoch 打印 loss/dev acc/macro recall；best 按 valdev macro recall 选轮）
!cd /kaggle/working/scripts && python train_action.py --arch both --data-root {DATA} --work-dir /kaggle/working/models/exp5_action --device cuda:0 --epochs 20 --batch-size 32 --num-workers 4

In [ ]:
# 导出：mc3_18 含 INT8 对比与 CPU 延迟（预期 INT8 反优化，负结果进报告）+ 部署 meta；r2plus1d_18 只 FP32
!cd /kaggle/working/scripts && python export_onnx.py --ckpt /kaggle/working/models/exp5_action/mc3_18/best.pt --emit-worker-meta /kaggle/working/action_clf_meta.json
!cd /kaggle/working/scripts && python export_onnx.py --ckpt /kaggle/working/models/exp5_action/r2plus1d_18/best.pt --no-int8

In [ ]:
# 打包回传物 → 右侧 Output 下载 action_train_return.zip（约 350MB）
import json
import shutil
from pathlib import Path

W = Path('/kaggle/working')
M = W / 'models/exp5_action'
OUT = W / 'action_train_return'
OUT.mkdir(exist_ok=True)
picks = [
    M / 'mc3_18/best.pt', M / 'r2plus1d_18/best.pt',
    M / 'mc3_18/action_mc3_18.onnx', M / 'r2plus1d_18/action_r2plus1d_18.onnx',
    W / 'action_clf_meta.json',
    M / 'mc3_18/train_summary.json', M / 'r2plus1d_18/train_summary.json',
    M / 'mc3_18/results.csv', M / 'r2plus1d_18/results.csv',
    M / 'mc3_18/export_meta.json', M / 'r2plus1d_18/export_meta.json',
    M / 'train_summary_all.json', M / 'data_meta_used.json',
]
for p in picks:
    if p.exists():
        shutil.copy2(p, OUT / p.name)
    else:
        print('缺失（可能训练中断）:', p)
for arch in ('mc3_18', 'r2plus1d_18'):
    f = M / arch / 'train_summary.json'
    if f.exists():
        s = json.loads(f.read_text())['test']
        print(f"{arch} 官方val: acc={s['accuracy']} macro_recall={s['macro_recall']}")
print('回传清单:', sorted(x.name for x in OUT.iterdir()))
shutil.make_archive(str(W / 'action_train_return'), 'zip', root_dir=W, base_dir='action_train_return')
print('>> 下载 /kaggle/working/action_train_return.zip，按 docs/实验5_动作识别执行计划.md §3 部署验收')